# Wellbeing Coach RAG App — LangChain + Pinecone + LangGraph

**Corpus:** *Dance To Your Maximum* by Maximiliaan Winkelhuis  
**Stack:** LangChain · LangGraph · OpenAI (`gpt-4.1-mini`, `text-embedding-3-small`) · Pinecone · LangSmith · Streamlit  

## Architecture
```
PDF ──► PyPDFLoader ──► Cleaner ──► Structure Detector ──► Hierarchical Chunker
                                                                    │
                                                     OpenAIEmbeddings (text-embedding-3-small)
                                                                    │
                                                            Pinecone (cosine)

User Question ──► LangGraph:
                  [route] → [retrieve] → [generate] ──► Answer + inline citations
                                                                    │
                                                          Streamlit Chat UI
```

## Notebook Sections
1. Setup & Dependencies
2. Environment & LangSmith
3. PDF Ingestion
4. Text Cleaning
5. Structure Detection & Metadata
6. Hierarchical Chunking
7. Pinecone Indexing (run once)
8. Query Router
9. RAG Chain — LangGraph
10. Interactive Test
11. Faithfulness Evaluation

## 1. Setup & Dependencies

Before any code can run, we need a clean Python environment with every library the project depends on, and a Jupyter kernel that points to that environment.

In [ ]:
# Run once to install dependencies
# %pip install langchain langchain-openai langchain-pinecone langchain-community
# %pip install langgraph langsmith pinecone-client pypdf python-dotenv streamlit

## 2. Environment & LangSmith

API keys and project settings live in a `.env` file so they are never hard-coded in the notebook. `python-dotenv` loads them into memory at runtime. LangSmith tracing is activated automatically once the keys are present.

In [ ]:
import os
import re
import json
import io
import hashlib
from pathlib import Path
from typing import TypedDict, List

import fitz                         # pymupdf — renders PDF pages to images
from PIL import Image
import pytesseract                  # OCR engine wrapper

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import BaseMessage
from langgraph.graph import StateGraph, END
from pinecone import Pinecone, ServerlessSpec

load_dotenv()

OPENAI_API_KEY    = os.getenv('OPENAI_API_KEY')
PINECONE_API_KEY  = os.getenv('PINECONE_API_KEY')
LANGCHAIN_API_KEY = os.getenv('LANGCHAIN_API_KEY')

print(f"LangSmith project : {os.getenv('LANGCHAIN_PROJECT')}")
print(f"Tracing enabled   : {os.getenv('LANGCHAIN_TRACING_V2')}")
print(f"OpenAI key set    : {bool(OPENAI_API_KEY)}")
print(f"Pinecone key set  : {bool(PINECONE_API_KEY)}")

## 3. PDF Ingestion — OCR Pipeline

*Dance To Your Maximum* is a scanned workbook — every page is a photograph of text with no selectable characters. Standard PDF loaders return blank pages. We use **PyMuPDF** to render each page to a grayscale image and **Tesseract** to read the text via OCR. Results are cached to disk so the slow OCR step runs only once.

In [ ]:
from IPython.display import HTML

HTML("""
<div style="font-family:'Segoe UI',Arial,sans-serif;background:#f5f3ff;padding:28px 32px;border-radius:18px;max-width:960px;box-sizing:border-box;">

  <!-- Title -->
  <h2 style="margin:0 0 4px;font-size:1.75rem;color:#1e1b4b;">PDF Ingestion &mdash; OCR Pipeline</h2>
  <p style="margin:0 0 24px;color:#6b7280;font-size:0.95rem;">Turn a scanned, image-based PDF into LangChain Documents using PyMuPDF + Tesseract</p>

  <!-- 3 Step Cards -->
  <div style="display:flex;align-items:flex-start;gap:10px;margin-bottom:24px;">

    <!-- Step 1 -->
    <div style="flex:1;background:#fff;border:2px solid #c4b5fd;border-radius:14px;padding:16px;">
      <div style="display:flex;align-items:center;gap:8px;margin-bottom:12px;">
        <span style="background:#7c3aed;color:#fff;border-radius:50%;min-width:26px;height:26px;display:inline-flex;align-items:center;justify-content:center;font-weight:700;font-size:0.85rem;">1</span>
        <strong style="color:#1e1b4b;font-size:0.95rem;">Load PDF</strong>
      </div>
      <div style="background:#faf5ff;border-radius:8px;padding:10px;font-size:0.82rem;color:#374151;margin-bottom:10px;">
        📄 <em>e-Book_dance-to-your-maximum.pdf</em><br><br>
        <span style="background:#ede9fe;color:#5b21b6;padding:2px 8px;border-radius:20px;font-size:0.78rem;">316 pages</span>&nbsp;
        <span style="background:#fee2e2;color:#991b1b;padding:2px 8px;border-radius:20px;font-size:0.78rem;">image-based</span>
      </div>
      <div style="font-size:0.8rem;color:#6b7280;line-height:1.6;">
        <b>Tool:</b> PyMuPDF &mdash; <code>fitz.open()</code><br>
        <b>Why:</b> PyPDFLoader returns empty text &mdash; the PDF has no embedded text layer (scanned pages only)
      </div>
    </div>

    <!-- Arrow -->
    <div style="font-size:1.6rem;color:#7c3aed;padding-top:28px;">&#8594;</div>

    <!-- Step 2 -->
    <div style="flex:1;background:#fff;border:2px solid #c4b5fd;border-radius:14px;padding:16px;">
      <div style="display:flex;align-items:center;gap:8px;margin-bottom:12px;">
        <span style="background:#7c3aed;color:#fff;border-radius:50%;min-width:26px;height:26px;display:inline-flex;align-items:center;justify-content:center;font-weight:700;font-size:0.85rem;">2</span>
        <strong style="color:#1e1b4b;font-size:0.95rem;">OCR Each Page</strong>
      </div>
      <div style="background:#faf5ff;border-radius:8px;padding:10px;font-size:0.82rem;color:#374151;margin-bottom:10px;line-height:1.8;">
        🖼️ Render page &rarr; grayscale image (150 DPI)<br>
        🔍 PIL Image &rarr; Tesseract OCR<br>
        📝 Output: extracted text string
      </div>
      <div style="font-size:0.8rem;color:#6b7280;line-height:1.6;">
        <b>Cache:</b> saved to <code>data/ocr_cache.json</code><br>
        <b>On rerun:</b> loads from cache instantly &mdash; skips OCR entirely
      </div>
    </div>

    <!-- Arrow -->
    <div style="font-size:1.6rem;color:#7c3aed;padding-top:28px;">&#8594;</div>

    <!-- Step 3 -->
    <div style="flex:1;background:#fff;border:2px solid #c4b5fd;border-radius:14px;padding:16px;">
      <div style="display:flex;align-items:center;gap:8px;margin-bottom:12px;">
        <span style="background:#7c3aed;color:#fff;border-radius:50%;min-width:26px;height:26px;display:inline-flex;align-items:center;justify-content:center;font-weight:700;font-size:0.85rem;">3</span>
        <strong style="color:#1e1b4b;font-size:0.95rem;">Wrap as LangChain Document</strong>
      </div>
      <div style="background:#faf5ff;border-radius:8px;padding:10px;font-size:0.82rem;color:#374151;margin-bottom:10px;">
        <b>LangChain Document</b><br>
        <span style="color:#7c3aed;">page_content</span><br>
        &nbsp;&nbsp;OCR-extracted text<br><br>
        <span style="color:#7c3aed;">metadata</span><br>
        &nbsp;&nbsp;<span style="background:#ede9fe;padding:1px 6px;border-radius:4px;font-size:0.77rem;">source</span>&nbsp;
        <span style="background:#ede9fe;padding:1px 6px;border-radius:4px;font-size:0.77rem;">page</span>&nbsp;
        <span style="background:#ede9fe;padding:1px 6px;border-radius:4px;font-size:0.77rem;">page_num</span>
      </div>
      <div style="font-size:0.8rem;color:#6b7280;">
        <b>Result:</b> list of <b>316 Document objects</b><br>
        page_num is 1-indexed for human-readable citations
      </div>
    </div>

  </div>

  <!-- Bottom row: code + tip -->
  <div style="display:flex;align-items:flex-start;gap:16px;">

    <!-- Code snippet -->
    <div style="flex:2;background:#1e1b4b;border-radius:12px;padding:16px;">
      <pre style="margin:0;color:#e2e8f0;font-size:0.8rem;line-height:1.7;overflow:auto;"><span style="color:#a78bfa;">for</span> i <span style="color:#a78bfa;">in</span> <span style="color:#34d399;">range</span>(len(doc)):
    pix  = doc[i].get_pixmap(matrix=mat, colorspace=fitz.csGRAY)
    img  = Image.frombytes(<span style="color:#fcd34d;">'L'</span>, (pix.width, pix.height), pix.samples)
    text = pytesseract.image_to_string(img, lang=<span style="color:#fcd34d;">'eng'</span>)
    raw_pages.append(Document(
        page_content=text,
        metadata={<span style="color:#fcd34d;">'source'</span>: <span style="color:#fcd34d;">'dance_to_your_maximum'</span>, <span style="color:#fcd34d;">'page_num'</span>: i+<span style="color:#34d399;">1</span>}
    ))</pre>
    </div>

    <!-- Tip box -->
    <div style="flex:1;background:#fff;border:2px solid #c4b5fd;border-radius:12px;padding:14px;font-size:0.82rem;color:#374151;line-height:1.7;">
      <div style="font-size:1.4rem;margin-bottom:6px;">🤖</div>
      OCR runs <b>once</b> then caches to disk.<br><br>
      On every subsequent run, the 316 pages load from <code>ocr_cache.json</code> in seconds &mdash; no GPU or API cost.
    </div>

  </div>

</div>
""")

In [ ]:
PDF_PATH   = r'data/e-Book_dance-to-your-maximum.pdf'
CACHE_PATH = Path('data/ocr_cache.json')

# Tesseract binary path -- update for your OS:
# Windows default: r'C:\Program Files\Tesseract-OCR\tesseract.exe'
# macOS Homebrew:  '/usr/local/bin/tesseract'
# Linux:           '/usr/bin/tesseract'
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'
def ocr_pdf(pdf_path: str, cache_path: Path, dpi: int = 150) -> list:
    """Render each PDF page to a grayscale image and run Tesseract OCR.
    Results are cached to disk so this only runs once."""

    if cache_path.exists():
        print(f'Cache found — loading from {cache_path}')
        with open(cache_path, 'r', encoding='utf-8') as f:
            cached = json.load(f)
        pages = [
            Document(page_content=c['text'], metadata=c['metadata'])
            for c in cached
        ]
        non_empty = sum(1 for p in pages if p.page_content.strip())
        print(f'Loaded {len(pages)} pages ({non_empty} with text) from cache.')
        return pages

    doc   = fitz.open(pdf_path)
    total = len(doc)
    print(f'Starting OCR on {total} pages at {dpi} DPI (grayscale).')
    print('Expect ~5–15 minutes depending on CPU. Progress shown every 20 pages.')

    pages = []
    cache = []
    mat   = fitz.Matrix(dpi / 72, dpi / 72)

    for i in range(total):
        page = doc[i]
        pix  = page.get_pixmap(matrix=mat, colorspace=fitz.csGRAY)
        img  = Image.frombytes('L', (pix.width, pix.height), pix.samples)
        text = pytesseract.image_to_string(img, lang='eng')

        metadata = {
            'source':   'dance_to_your_maximum',
            'page':     i,
            'page_num': i + 1,
        }
        pages.append(Document(page_content=text, metadata=metadata))
        cache.append({'text': text, 'metadata': metadata})

        if (i + 1) % 20 == 0 or i == total - 1:
            print(f'  {i + 1:>3}/{total} pages done')

    doc.close()

    cache_path.parent.mkdir(parents=True, exist_ok=True)
    with open(cache_path, 'w', encoding='utf-8') as f:
        json.dump(cache, f, ensure_ascii=False)
    print(f'\nOCR complete. Cache saved → {cache_path}')

    return pages

raw_pages = ocr_pdf(PDF_PATH, CACHE_PATH)

# Quick verification
non_empty = [p for p in raw_pages if p.page_content.strip()]
print(f'\nPages with text: {len(non_empty)} / {len(raw_pages)}')
if non_empty:
    print(f'\nSample — page {non_empty[0].metadata["page_num"]}:')
    print(non_empty[0].page_content[:400])

## 4. Text Cleaning

OCR output contains artefacts: words split across lines with a dash, extra spaces, and strings of blank lines. We fix these conservatively — never removing chapter numbers, section headings, or page markers that are needed later for metadata and citations.

In [ ]:
def clean_text(text: str) -> str:
    # Merge hyphenated line breaks: "danc-\ning" → "dancing"
    text = re.sub(r'(\w)-\n(\w)', r'\1\2', text)
    # Collapse multiple spaces/tabs to a single space
    text = re.sub(r'[ \t]+', ' ', text)
    # Collapse 3+ blank lines to a paragraph break
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

pages_cleaned = []
for page in raw_pages:
    page.page_content = clean_text(page.page_content)
    pages_cleaned.append(page)

print(f'Cleaned {len(pages_cleaned)} pages')

## 5. Structure Detection & Metadata Enrichment

The book has three Parts, numbered chapters (e.g. `1-2`, `2-8`), and distinct section types (prose text, personal tests, evaluation forms, appendices). We scan each page with regex patterns to detect structural markers and attach rich metadata that travels with every chunk into Pinecone, enabling precise filtered retrieval later.

In [ ]:
PART_PATTERNS = [
    (re.compile(r'\bpart\s+one\b', re.IGNORECASE),   'Part One',   'competition_day'),
    (re.compile(r'\bpart\s+two\b', re.IGNORECASE),   'Part Two',   'season'),
    (re.compile(r'\bpart\s+three\b', re.IGNORECASE), 'Part Three', 'career'),
]

CHAPTER_PATTERN = re.compile(r'\b(\d+-\d+)\s+([^\n]{5,60})', re.IGNORECASE)

SECTION_KEYWORDS = {
    'toc':      [re.compile(r'\btable\s+of\s+contents\b', re.IGNORECASE)],
    'intro':    [re.compile(r'\bintroduction\b', re.IGNORECASE),
                 re.compile(r'\bhow\s+to\s+use\s+this\s+book\b', re.IGNORECASE),
                 re.compile(r'\bforeword\b', re.IGNORECASE),
                 re.compile(r'\bpreface\b', re.IGNORECASE)],
    'test':     [re.compile(r'\bquestionnaire\b', re.IGNORECASE),
                 re.compile(r'\bself[-\s]?assessment\b', re.IGNORECASE),
                 re.compile(r'\bpersonal\s+test\b', re.IGNORECASE)],
    'form':     [re.compile(r'\bevaluation\s+form\b', re.IGNORECASE),
                 re.compile(r'\bworksheet\b', re.IGNORECASE)],
    'appendix': [re.compile(r'\bappendix\b', re.IGNORECASE)],
}

def detect_section_type(text: str) -> str:
    for stype, patterns in SECTION_KEYWORDS.items():
        for p in patterns:
            if p.search(text):
                return stype
    return 'prose'

def enrich_pages(pages: list) -> list:
    current_part        = ''
    current_part_scope  = ''
    current_chapter     = ''
    current_ch_title    = ''

    enriched = []
    for page in pages:
        text     = page.page_content
        page_num = page.metadata.get('page_num', 0)

        for pattern, name, scope in PART_PATTERNS:
            if pattern.search(text):
                current_part       = name
                current_part_scope = scope
                break

        ch_match = CHAPTER_PATTERN.search(text)
        if ch_match:
            current_chapter  = ch_match.group(1)
            current_ch_title = ch_match.group(2).strip()

        section_type = detect_section_type(text)

        page.metadata.update({
            'source':        'dance_to_your_maximum',
            'part':          current_part,
            'part_scope':    current_part_scope,
            'chapter':       current_chapter,
            'chapter_title': current_ch_title,
            'section_type':  section_type,
            'page_start':    page_num,
            'page_end':      page_num,
        })
        enriched.append(page)

    return enriched

enriched_pages = enrich_pages(pages_cleaned)

print('Part distribution:')
from collections import Counter
print(Counter(p.metadata['part'] for p in enriched_pages))
print('\nSection type distribution:')
print(Counter(p.metadata['section_type'] for p in enriched_pages))

## 6. Hierarchical Chunking

Long pages are split into retrieval-sized chunks. We use different size rules depending on section type: prose chapters get smaller chunks for precision, while forms and questionnaires get larger chunks so that questions and answer spaces stay together. Chunks that are too small (blank pages, isolated headers) are discarded.

In [ ]:
# ── DIAGNOSTIC: check page content before chunking ───────────────────────────
total        = len(enriched_pages)
non_empty    = [(p.metadata.get('page_num', i), len(p.page_content))
                for i, p in enumerate(enriched_pages)
                if len(p.page_content.strip()) > 10]

print(f'Total pages loaded : {total}')
print(f'Pages with text    : {len(non_empty)}')

if non_empty:
    sizes = [sz for _, sz in non_empty]
    print(f'Avg page length    : {sum(sizes) // len(sizes)} chars')
    print(f'Min page length    : {min(sizes)} chars')
    print(f'Max page length    : {max(sizes)} chars')
    print(f'\nSample — page {non_empty[0][0]}:')
    first_page = next(p for p in enriched_pages if len(p.page_content.strip()) > 10)
    print(first_page.page_content[:500])
else:
    print('\nWARNING: ALL pages have empty text.')
    print('The PDF is likely image-based (scanned). OCR will be needed.')
    print('Raw metadata of page 1:', enriched_pages[0].metadata if enriched_pages else 'No pages')

In [ ]:
PROSE_SPLITTER = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=200,
    separators=['\n\n', '\n', '. ', ' ', ''],
    length_function=len,
)

FORM_SPLITTER = RecursiveCharacterTextSplitter(
    chunk_size=2200,
    chunk_overlap=150,
    separators=['\n\n', '\n', ' ', ''],
    length_function=len,
)

# Lowered from 350 → 100: workbook pages (forms, bullet lists, blank answer lines)
# produce many short chunks that are still semantically meaningful.
MIN_CHUNK_SIZE   = 100
FORM_SECTION_TYPES = {'test', 'form', 'appendix'}

def chunk_pages(pages: list) -> list:
    all_chunks = []
    skipped    = 0
    raw_total  = 0

    for page in pages:
        section_type = page.metadata.get('section_type', 'prose')
        splitter     = FORM_SPLITTER if section_type in FORM_SECTION_TYPES else PROSE_SPLITTER
        parent_id    = f"page_{page.metadata.get('page_start', 0)}"

        raw_chunks = splitter.split_documents([page])
        raw_total += len(raw_chunks)

        for chunk in raw_chunks:
            if len(chunk.page_content) < MIN_CHUNK_SIZE:
                skipped += 1
                continue
            chunk_id = hashlib.md5(chunk.page_content.encode()).hexdigest()[:12]
            chunk.metadata['chunk_id']  = chunk_id
            chunk.metadata['parent_id'] = parent_id
            all_chunks.append(chunk)

    print(f'Raw chunks created : {raw_total}')
    print(f'Kept (≥{MIN_CHUNK_SIZE} chars): {len(all_chunks)}')
    print(f'Skipped            : {skipped}')

    if all_chunks:
        sizes = [len(c.page_content) for c in all_chunks]
        print(f'\nChunk size stats:')
        print(f'  min  : {min(sizes)}')
        print(f'  max  : {max(sizes)}')
        print(f'  mean : {sum(sizes)//len(sizes)}')

    return all_chunks

chunks = chunk_pages(enriched_pages)

if chunks:
    print('\nSample chunk:')
    print(chunks[0].page_content[:300])
    print('\nMetadata:', chunks[0].metadata)
else:
    print('\nWARNING: chunks is still empty.')
    print('Page count        :', len(enriched_pages))
    print('Pages with content:', sum(1 for p in enriched_pages if len(p.page_content.strip()) > 0))
    print('\nFirst non-empty page content:')
    for p in enriched_pages:
        if p.page_content.strip():
            print(p.page_content[:400])
            break

## 7. Pinecone Indexing

Each chunk is converted into a 1536-dimension vector by OpenAI's `text-embedding-3-small` model — numbers that capture the semantic meaning of the text. These vectors are stored in Pinecone with all their metadata attached. **Run this section once only.** After the initial upload, comment out the upsert call to avoid re-indexing.

In [ ]:
INDEX_NAME    = 'wellbeing-coach-rag'
EMBEDDING_DIM = 1536

pc = Pinecone(api_key=PINECONE_API_KEY)

existing = [idx.name for idx in pc.list_indexes()]
if INDEX_NAME not in existing:
    pc.create_index(
        name=INDEX_NAME,
        dimension=EMBEDDING_DIM,
        metric='cosine',
        spec=ServerlessSpec(cloud='aws', region='us-east-1'),
    )
    print(f'Created index: {INDEX_NAME}')
else:
    print(f'Index already exists: {INDEX_NAME}')

embeddings  = OpenAIEmbeddings(model='text-embedding-3-small')
vectorstore = PineconeVectorStore(index_name=INDEX_NAME, embedding=embeddings)

print(f'Vectorstore ready: {INDEX_NAME}')

In [ ]:
# ── INITIAL INGESTION — comment out after first run ─────────────────────────

BATCH_SIZE = 100

def upsert_chunks(chunks: list, vectorstore: PineconeVectorStore, batch_size: int = 100):
    total = len(chunks)
    for i in range(0, total, batch_size):
        batch = chunks[i : i + batch_size]
        vectorstore.add_documents(batch)
        print(f'  Upserted {min(i + batch_size, total):>4}/{total}')
    print('Ingestion complete.')

upsert_chunks(chunks, vectorstore, BATCH_SIZE)

## 8. Query Router

Not all questions need to search the entire book. A competition-day stress question should focus on Part One; a season-planning question on Part Two. The router uses `gpt-4.1-mini` to classify each incoming question and returns a routing dictionary that the retriever uses to apply metadata filters in Pinecone.

In [ ]:
# ── PRE-LAUNCH CHECK: verify Pinecone index has vectors ──────────────────────
pc_check  = Pinecone(api_key=PINECONE_API_KEY)
idx_stats = pc_check.Index('wellbeing-coach-rag').describe_index_stats()
vector_count = idx_stats.get('total_vector_count', 0)

print(f'Vectors in index : {vector_count}')
if vector_count > 0:
    print('Index is ready. You can launch the Streamlit app.')
    print('\nRun in terminal:')
    print('  .venv\\Scripts\\streamlit.exe run app.py')
else:
    print('Index is empty — run Section 7 (Pinecone Indexing) first.')

In [ ]:
ROUTER_PROMPT = ChatPromptTemplate.from_messages([
    ('system', '''Classify the user\'s DanceSport query into routing categories.

part_scope options:
- competition_day: competition day stress, morning routine, warm-up, floor performance
- season: season planning, choreography, exercises, training load
- career: long-term goals, career arc, identity, calling
- general: spans multiple parts or unclear

section_type options:
- test: specific test, questionnaire, or self-assessment
- form: evaluation form or worksheet
- appendix: appendix content
- prose: regular explanatory content
- general: unclear

Return valid JSON only. Example: {{"part_scope": "competition_day", "section_type": "prose"}}'''),
    ('human', 'Query: {question}'),
])

llm_router = ChatOpenAI(model='gpt-4.1-mini', temperature=0)

def route_query(question: str) -> dict:
    chain  = ROUTER_PROMPT | llm_router | StrOutputParser()
    result = chain.invoke({'question': question})
    try:
        return json.loads(result)
    except json.JSONDecodeError:
        return {'part_scope': 'general', 'section_type': 'general'}

# Smoke test
print(route_query('How do I manage stress on competition morning?'))
print(route_query('Where is the Nine Step Connection Model questionnaire?'))
print(route_query('How do I plan my long-term career as a dancer?'))

## 9. RAG Chain — LangGraph

The full pipeline is wired as a stateful three-node **LangGraph** graph. State (the question, routing decision, retrieved chunks, and final answer) flows from node to node. Each node does one job: route → retrieve → generate. This makes the pipeline easy to inspect, test each node independently, and extend later.

In [ ]:
SYSTEM_PROMPT = '''You are a Wellbeing Coach who helps DanceSport athletes answer questions on competition prep and performance readiness. Answer using ONLY the provided context. If you cannot find the answer, say \'I don\'t have that in my knowledge base.\' Do NOT hallucinate.

You are top expert in the domains at the intersection of DanceSport, High Performance, Wellbeing. Accuracy beats approval. Blunt, argumentative. No disclaimers or praise. Lead with counterarguments. Don\'t capitulate without new evidence.

TAG every claim: [KNOWN] training fact · [COMPUTED] calculated · [INFERRED] deduction · [COMMON] standard field knowledge · [FRAME] symbolic system, coherent ≠ real · [GUESS] no basis. No untagged disease, statute, citation, or named entity.

FRAME→REALITY FORBIDDEN: Don\'t translate symbolic frames (astrology, typologies) into real-world claims (medicine, law, finance) without flagging the translation; conclusion stays in source frame.

CONFIDENCE: HIGH ≥80% · MED 50–80% · LOW 20–50% · VERY LOW <20% · UNKNOWN. [FRAME] real-world and [GUESS] cap at LOW.

DON\'T KNOW: First line "I don\'t know." Don\'t bury, don\'t fabricate.

ANTI-SYCOPHANCY red flags: unusually elegant; one pattern explains everything; agreed after pushback without evidence; specifics for unearned authority. Fire → cut specifics, add [GUESS], or "I don\'t know."

POST-HOC: Would the frame predict this without knowing the outcome? If no: [INFERRED, post-hoc], accommodates, doesn\'t predict.

Never fabricate citations. Revise openly if holding a position for consistency. Append "[RULES I BROKE]: which, where, why."

CITATION FORMAT: After every claim, add an inline citation: [Dance To Your Maximum, Chapter X-Y, Page Z–W]
Use the chapter and page metadata from the provided context documents.'''

In [ ]:
RAG_PROMPT = ChatPromptTemplate.from_messages([
    ('system', SYSTEM_PROMPT),
    ('human', 'Context from knowledge base:\n\n{context}\n\nQuestion: {question}'),
])

llm_answer = ChatOpenAI(model='gpt-4.1-mini', temperature=0.1)


class RAGState(TypedDict):
    question:     str
    chat_history: List[BaseMessage]
    route:        dict
    documents:    List[Document]
    answer:       str


def build_filter(route: dict) -> dict:
    filters     = {}
    part_scope  = route.get('part_scope', 'general')
    section_type = route.get('section_type', 'general')
    if part_scope and part_scope != 'general':
        filters['part_scope'] = {'$eq': part_scope}
    if section_type and section_type not in {'prose', 'general'}:
        filters['section_type'] = {'$in': [section_type]}
    return filters


def format_context(docs: List[Document]) -> str:
    parts = []
    for doc in docs:
        meta       = doc.metadata
        chapter    = meta.get('chapter', 'N/A')
        page_start = meta.get('page_start', '')
        page_end   = meta.get('page_end', '')
        if page_start and page_end and str(page_start) != str(page_end):
            cite = f'[Dance To Your Maximum, Chapter {chapter}, Page {page_start}\u2013{page_end}]'
        elif page_start:
            cite = f'[Dance To Your Maximum, Chapter {chapter}, Page {page_start}]'
        else:
            cite = f'[Dance To Your Maximum, Chapter {chapter}]'
        parts.append(f'{doc.page_content}\n\nSource: {cite}')
    return '\n\n---\n\n'.join(parts)


def route_node(state: RAGState) -> dict:
    return {'route': route_query(state['question'])}


def retrieve_node(state: RAGState) -> dict:
    route       = state.get('route', {})
    filter_dict = build_filter(route)
    if filter_dict:
        docs = vectorstore.similarity_search(state['question'], k=4, filter=filter_dict)
        if not docs:
            docs = vectorstore.similarity_search(state['question'], k=6)
    else:
        docs = vectorstore.similarity_search(state['question'], k=6)
    return {'documents': docs}


def generate_node(state: RAGState) -> dict:
    docs = state.get('documents', [])
    if not docs:
        return {'answer': "I don't have that in my knowledge base."}
    context = format_context(docs)
    chain   = RAG_PROMPT | llm_answer | StrOutputParser()
    answer  = chain.invoke({'context': context, 'question': state['question']})
    return {'answer': answer}


def build_rag_graph():
    graph = StateGraph(RAGState)
    graph.add_node('route',    route_node)
    graph.add_node('retrieve', retrieve_node)
    graph.add_node('generate', generate_node)
    graph.set_entry_point('route')
    graph.add_edge('route',    'retrieve')
    graph.add_edge('retrieve', 'generate')
    graph.add_edge('generate', END)
    return graph.compile()


rag_app = build_rag_graph()
print('RAG graph compiled.')

## 10. Interactive Test

Send live questions through the complete pipeline and read the output. Every answer should be tagged (`[KNOWN]`, `[INFERRED]`, etc.) and include inline citations pointing to the exact chapter and page. Test at least one question per Part to confirm routing and retrieval are working correctly across the whole book.

In [ ]:
def ask(question: str, chat_history: list = None) -> str:
    result = rag_app.invoke({
        'question':     question,
        'chat_history': chat_history or [],
        'route':        {},
        'documents':    [],
        'answer':       '',
    })
    return result['answer']


# ── Test queries ─────────────────────────────────────────────────────────────
q1 = 'How should I manage pre-competition stress on competition morning?'
print(f'Q: {q1}')
print(f'A: {ask(q1)}')

In [ ]:
q2 = 'What is the Nine Step Connection Model?'
print(f'Q: {q2}')
print(f'A: {ask(q2)}')

In [ ]:
q3 = 'What does the book say about long-term career planning for dancers?'
print(f'Q: {q3}')
print(f'A: {ask(q3)}')

## 11. Faithfulness Evaluation

Faithfulness measures whether every factual claim in the answer is directly supported by the retrieved passages — it is not a measure of real-world accuracy. We run a fixed 15-question benchmark, invoke the RAG chain for each, and ask `gpt-4.1-mini` to judge each answer. The target is **≥ 90% faithful answers**.

In [ ]:
EVAL_QUESTIONS = [
    'How should I manage pre-competition stress on competition day?',
    'What is the Nine Step Connection Model?',
    'How do I plan my choreography during the season?',
    'What exercises are recommended during the season?',
    'How do I plan my DanceSport career long-term?',
    'What does the book say about the Autonomous Nervous System and stress?',
    'How can I use simulation to prepare for competition?',
    'What is the recommended morning routine before a competition?',
    'How do I evaluate my own performance after a competition?',
    'What does the book say about goal setting for dancers?',
    'How should I warm up before going on the competition floor?',
    'What is the role of mental preparation in DanceSport?',
    'How do I maintain peak condition throughout the season?',
    'What does the book say about recovery after competitions?',
    'How do I handle judging criteria and feedback from judges?',
]

JUDGE_PROMPT = ChatPromptTemplate.from_messages([
    ('system', '''You are an expert evaluator assessing faithfulness of AI-generated answers.
Faithfulness means: every factual claim in the answer is directly supported by the provided context.
If the answer says "I don\'t have that in my knowledge base", it is FAITHFUL (correct refusal).
Respond with exactly: FAITHFUL or NOT_FAITHFUL, then a colon and a brief one-line reason.
Example: FAITHFUL: All claims trace to the provided context.'''),
    ('human', 'Context:\n{context}\n\nQuestion: {question}\n\nAnswer: {answer}'),
])

llm_judge = ChatOpenAI(model='gpt-4.1-mini', temperature=0)


def evaluate_faithfulness(questions: list, n: int = 15) -> dict:
    results = []
    print(f'Evaluating {min(n, len(questions))} questions...\n')

    for i, q in enumerate(questions[:n], 1):
        state = rag_app.invoke({
            'question':     q,
            'chat_history': [],
            'route':        {},
            'documents':    [],
            'answer':       '',
        })
        docs    = state.get('documents', [])
        answer  = state.get('answer', '')
        context = format_context(docs) if docs else 'No context retrieved.'

        judge_chain = JUDGE_PROMPT | llm_judge | StrOutputParser()
        verdict     = judge_chain.invoke({'context': context, 'question': q, 'answer': answer})
        is_faithful = verdict.upper().startswith('FAITHFUL')

        results.append({
            'question':   q,
            'answer':     answer[:200] + '...' if len(answer) > 200 else answer,
            'verdict':    verdict,
            'faithful':   is_faithful,
        })
        symbol = '\u2713' if is_faithful else '\u2717'
        print(f'[{i:>2}] {symbol} {q[:65]}')
        if not is_faithful:
            print(f'       {verdict}')

    faithful_count = sum(1 for r in results if r['faithful'])
    score          = faithful_count / len(results) if results else 0.0

    print(f'\n{"─" * 60}')
    print(f'Faithfulness: {faithful_count}/{len(results)} = {score:.1%}')
    print(f'Target       : 90%')
    print(f'Result       : {"PASS" if score >= 0.9 else "FAIL"}')

    return {'results': results, 'score': score, 'pass': score >= 0.9}


eval_results = evaluate_faithfulness(EVAL_QUESTIONS)

In [ ]:
# Detailed failure review
failures = [r for r in eval_results['results'] if not r['faithful']]
if failures:
    print(f'{len(failures)} failed questions:\n')
    for f in failures:
        print(f'Q  : {f["question"]}')
        print(f'A  : {f["answer"]}')
        print(f'WHY: {f["verdict"]}')
        print()
else:
    print('All questions passed faithfulness check.')